# 07 — Negative / correction clicks (inference only)

**Hypothesis.** Adding neighbour negative clicks at tile time raises F1 for a fixed checkpoint.

**Runtime.** T4 or L4 (~8–12 GB). No training.

Uses `configs/experiment/negative_clicks.yaml`.


In [ ]:
CHECKPOINT = "results/isprs_lora/best.safetensors"  # or best from 02/06
AUTO_SHUTDOWN = True


In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd().resolve()
for _ in range(4):
    if (root / "src" / "treeseg").is_dir():
        break
    root = root.parent
os.chdir(root)
sys.path.insert(0, "src")

from treeseg.engine.experiment_run import ensure_repo_root, probe_resources, maybe_shutdown, comparison_table
from treeseg.engine.benchmark import run_benchmark, summarise
from treeseg.models.point_sam import add_point_sam_to_path
from treeseg.utils.curves import plot_f1_bars
from IPython.display import Image, display
import yaml, torch

ensure_repo_root()
add_point_sam_to_path()
cfg = yaml.safe_load(open("configs/experiment/negative_clicks.yaml"))
cfg["checkpoints"] = {"run": CHECKPOINT}
assert Path(CHECKPOINT).is_file(), CHECKPOINT
os.makedirs(cfg["out_dir"], exist_ok=True)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
probe_resources("neg_start")
run_benchmark(cfg)
summary = summarise(os.path.join(cfg["out_dir"], "results.csv"))
summary.to_csv(os.path.join(cfg["out_dir"], "summary.csv"), index=False)
table = comparison_table(summary)
display(table)
plot_f1_bars(summary, os.path.join(cfg["out_dir"], "f1_bars.png"))
display(Image(os.path.join(cfg["out_dir"], "f1_bars.png")))
probe_resources("neg_end")
maybe_shutdown(AUTO_SHUTDOWN)
